In [1]:
import pandas as pd
import numpy as np

In [2]:
import os
os.getcwd()
os.chdir('a:\\Dominic\\My Projects\\Home Credit\\')

In [33]:
PrevLoans = pd.read_parquet("a:\\Dominic\\My Projects\\Home Credit\\src\\intermediate\\prev_loan_data_processed.parquet")

In [34]:
PrevLoans = PrevLoans.astype({'SK_ID_PREV': 'string', 'SK_ID_CURR': 'string'})

In [ ]:
POSBal = pd.read_csv('data/Internal/POS_CASH_balance.csv')
POSBal = POSBal.astype({'SK_ID_PREV': 'string', 'SK_ID_CURR': 'string'})
POSBal.sort_values(by=['SK_ID_CURR','SK_ID_PREV','MONTHS_BALANCE'], inplace=True, ignore_index=True)

In [6]:
base = pd.read_csv('data/Internal/application_train.csv', usecols=['SK_ID_CURR', 'TARGET'])
base = base.astype({'SK_ID_CURR': 'string'})

In [8]:
POSBal = base.merge(POSBal, on='SK_ID_CURR', how='inner').drop(columns=['TARGET'])

In [39]:
POSBal['CNT_INSTALMENT'] = POSBal.groupby(['SK_ID_PREV'])['CNT_INSTALMENT'].transform(
    lambda x: x.ffill(limit=3).bfill(limit=3)
)
POSBal['CNT_INSTALMENT_FUTURE'] = POSBal.groupby(['SK_ID_PREV'])['CNT_INSTALMENT_FUTURE'].transform(
    lambda x: x.ffill(limit=3).bfill(limit=3)
)

In [7]:
#check for installment prepayment

Aggregate to Monthly

In [27]:
POSBal_Mon = POSBal.groupby(['SK_ID_CURR','MONTHS_BALANCE'])[['CNT_INSTALMENT', 'CNT_INSTALMENT_FUTURE', 'SK_DPD', 'SK_DPD_DEF']].agg('max')

In [ ]:
POSBal_Mon.columns = [''.join(x).upper() for x in POSBal_Mon.columns]
POSBal_Mon = POSBal_Mon.reset_index()

In [45]:
pos_aggregated = POSBal_Mon.groupby('SK_ID_CURR').agg(
    POS_DAYS_IN_DPD=('SK_DPD', lambda x: sum((x > 0)*1)),
    POS_DAYS_IN_DPD_DEF=('SK_DPD_DEF', lambda x: sum((x > 0)*1)),
    POS_DAYS_IN_DPD_TO_DURATION_RATIO=('SK_DPD', lambda x: sum((x > 0)*1)/len(x) if len(x) > 0 else 0),
)

In [46]:
pos_aggregated_1Y = POSBal_Mon[POSBal_Mon['MONTHS_BALANCE'] >= -12].groupby('SK_ID_CURR').agg(
    POS_DAYS_IN_DPD_1Y=('SK_DPD', lambda x: sum((x > 0)*1)),
    POS_DAYS_IN_DPD_DEF_1Y=('SK_DPD_DEF', lambda x: sum((x > 0)*1)),
    POS_DAYS_IN_DPD_TO_DURATION_RATIO_1Y=('SK_DPD', lambda x: sum((x > 0)*1)/len(x) if len(x) > 0 else 0),
)

In [47]:
pos_aggregated_3M = POSBal_Mon[POSBal_Mon['MONTHS_BALANCE'] >= -3].groupby('SK_ID_CURR').agg(
    POS_DAYS_IN_DPD_3M=('SK_DPD', lambda x: sum((x > 0)*1)),
    POS_DAYS_IN_DPD_DEF_3M=('SK_DPD_DEF', lambda x: sum((x > 0)*1)),
    POS_DAYS_IN_DPD_TO_DURATION_RATIO_3M=('SK_DPD', lambda x: sum((x > 0)*1)/len(x) if len(x) > 0 else 0),
)

In [51]:
POSBal_first = POSBal.groupby(['SK_ID_CURR','SK_ID_PREV']).first().reset_index()
POSBal_last = POSBal.groupby(['SK_ID_CURR','SK_ID_PREV']).last().reset_index()

Prepayment Flag is actual duration < originalexpected duration

In [52]:
POSBal_CNT_INSTALMENT = (POSBal_first[[ 'SK_ID_CURR', 'SK_ID_PREV', 'CNT_INSTALMENT']]
           .merge(POSBal_last[[ 'SK_ID_CURR', 'SK_ID_PREV', 'CNT_INSTALMENT']], on=['SK_ID_CURR','SK_ID_PREV'], suffixes=('_FIRST', '_LAST'))
    )
POSBal_CNT_INSTALMENT['PREV_CREDIT_PREPAYMENT_FLAG'] = (POSBal_CNT_INSTALMENT['CNT_INSTALMENT_FIRST'] != POSBal_CNT_INSTALMENT['CNT_INSTALMENT_LAST'])*1

In [50]:
POSBal_Agg_last = POSBal_last.groupby('SK_ID_CURR').agg(
    POS_AVG_CNT_INSTALMENT_FINAL=('CNT_INSTALMENT', 'mean'),
    POS_AVG_CNT_INSTALMENT_FUTURE=('CNT_INSTALMENT_FUTURE', 'mean'),
    POS_MAX_CNT_INSTALMENT_FINAL=('CNT_INSTALMENT', 'max'),
    POS_MAX_CNT_INSTALMENT_FUTURE=('CNT_INSTALMENT_FUTURE', 'max'),
    POS_CNT_ACTIVE_PREV_CONTRACT_STATUS=('NAME_CONTRACT_STATUS', lambda x: sum((x == 'Active')*1))
)

Aggregate at Previous ID level

In [53]:
POSBal_PrevLoans = POSBal_CNT_INSTALMENT.merge(PrevLoans, on=['SK_ID_PREV', 'SK_ID_CURR'], how='left')

In [55]:
POSBal_PrevLoans['AMT_ANNUITY_TO_CREDIT_RATIO'] = POSBal_PrevLoans['AMT_ANNUITY'] / POSBal_PrevLoans['AMT_CREDIT']

In [56]:
POSBal_Agg_PrevLoans = POSBal_PrevLoans.groupby('SK_ID_CURR').agg(
    POS_TOTAL_PREV_LOAN = ('SK_ID_PREV', 'nunique'),   
    POS_PREV_CREDIT_PREPAYMENT_FLAG = ('PREV_CREDIT_PREPAYMENT_FLAG', 'max'),

    POS_AVG_CREDIT_TO_APPL = ('CREDIT_TO_APPL', 'mean'),

    POS_ANNUITY_TO_CREDIT_RATIO = ('AMT_ANNUITY_TO_CREDIT_RATIO', 'mean'),

    POS_CNT_HIGH_YIELD_GROUP = ('NAME_YIELD_GROUP', lambda x: ((x=='high')*1).sum()),
    POS_CNT_MIDDLE_YIELD_GROUP = ('NAME_YIELD_GROUP', lambda x: ((x=='middle')*1).sum()),
    POS_CNT_NFLAG_INSURED_ON_APPROVAL_PREVLOAN = ('NFLAG_INSURED_ON_APPROVAL', 'sum'),
    POS_AVG_DAYS_DECISION_PREVLOAN = ('DAYS_DECISION', 'mean')

)

In [14]:
POSBal_Gold.to_parquet("a:\\Dominic\\My Projects\\Home Credit\\src\\intermediate\\posdata_processed.parquet")